# Impedance ML Engineering — Session 07
## Explore latent matrix structure with PCA
**Core: 60 minutes · Classical ML · Optional extension: 20–40 minutes**

## Why this matters
Can low-dimensional matrix representations reveal structure, and what does an unusual reconstruction mean?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Fit imputation/scaling/PCA using training patients only.
- Inspect explained variance and reshape loadings.
- Compare PCA features with spatial summaries for prediction.
- Treat anomalies as inspection prompts rather than diagnosis.

## R-to-Python / prerequisite bridge
`prcomp` → `PCA`; loadings are `.components_` rows; centering/scaling and missing-value choices change the representation.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
PCA finds orthogonal directions of greatest input variance. It ignores the prediction target. Standardizing each matrix coordinate changes the question from absolute variance toward relative coordinate variance, so it changes how diagonal and off-diagonal values contribute. Component signs are arbitrary.

Reconstruction error measures what a representation fails to recover in the chosen scale. It is not a diagnostic probability. A model can reconstruct protocol artifacts beautifully and still predict the future poorly. Fit even unsupervised transformations on the permitted training patients.

Imports: `PCA` from `sklearn.decomposition`. Access `.components_`, `.explained_variance_ratio_`, `.transform`, and `.inverse_transform`.

## Task 1 — Flatten without losing identity (5 min)
Flatten current matrices to (264, 144), keeping row-aligned IDs and a mapping from flattened positions to contact pairs. Predict the number of columns and where (stim 1,record 2) lands. Keep all directed cells. Work with training and validation only.

## Task 2 — Fit training-only preprocessing (7 min)
Create a pipeline `SimpleImputer(strategy="median")`, `StandardScaler()`, `PCA(n_components=10,random_state=42)`. Fit on training rows; transform validation. Check finite transformed inputs and identical widths. Explain why treating masked cells as literal zero before PCA would distort geometry.

## Task 3 — Inspect retained variance (6 min)
Plot cumulative `explained_variance_ratio_` for 10 components. Explain covariance-PCA versus standardized correlation-PCA; the scaling choice changes diagonal versus off-diagonal influence. Record this choice before comparing models.

## Task 4 — Interpret score patterns (7 min)
Plot PC1 versus PC2 training scores colored separately by protocol, array, and current diagonal mean. Show validation points distinctly without fitting to them. State whether a separation might represent measurement settings or subject baseline, and why it is not proof of biological subtypes.

## Task 5 — Inspect loading maps (7 min)
Reshape the first two `.components_` rows into 12×12 maps with common limits and contact labels. Explain sign ambiguity. Check that the original feature order matches reshape order. Identify one broad spatial pattern rather than naming a clinical marker.

## Task 6 — Compare prediction representations (10 min)
Build a pipeline that adds Ridge(alpha=1) after PCA and score validation next-visit MAE. Compare with persistence and the original summary-feature ridge on the same rows. Keep component count fixed at 10 for this experiment; choosing it by validation is model selection. Explain why more explained input variance does not guarantee better target prediction.

## Task 7 — Inspect reconstruction residuals (8 min)
Reconstruct standardized validation inputs via PCA.inverse_transform of scores and compute mean squared residual per matrix in that standardized space. Rank the top three and plot input/mask/residual maps. Distinguish observed-cell behavior from imputed-cell residuals; report observed-cell residual as the main inspection score.

## Task 8 — Write the unsupervised conclusion (2 min)
Report component count, scaling, retained variance, validation predictive comparison, and one inspected unusual pattern. Do not call a high residual a device failure: there are no diagnostic labels in this fixture.

## Retrieval practice
Describe which pipeline stages learn across patients, and where each would be fitted inside grouped CV.

## Professional reasoning
1. Why can reconstruction error be dominated by a protocol shift?
2. What could go wrong when a loading image is reshaped with the wrong axis order?
3. How would you validate a proposed cluster when no external subtype labels exist?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Try training-only KMeans on PCA scores with k=2 or 3 and inspect patient/protocol composition. Treat silhouettes as geometric evidence rather than external validation.

## API references
[scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)